# SmartPantryAI — Food Ingredient YOLO Training
**Settings → Accelerator → GPU T4 x2** before running.

In [ ]:
# Cell 1 — GPU check
import torch, os
gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print(f'GPU     : {gpu}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
assert torch.cuda.is_available(), 'No GPU — go to Settings → Accelerator → GPU T4 x2'

In [ ]:
# Cell 2 — install ultralytics only (do NOT touch torch)
!pip install ultralytics -q

In [ ]:
# Cell 3 — locate dataset
import glob, os
candidates = glob.glob('/kaggle/input/*/data/dataset.yaml')
assert candidates, 'dataset.yaml not found — add the dataset in the Data panel on the right'
dataset_root = os.path.dirname(os.path.dirname(candidates[0]))
print(f'Dataset root : {dataset_root}')
print(f'Total files  : {len(glob.glob(dataset_root + "/**/*", recursive=True))}')

In [ ]:
# Cell 4 — patch data.yaml with Kaggle input paths
import yaml
with open(f'{dataset_root}/data/dataset.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['path'] = f'{dataset_root}/data'
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(cfg, f)
print(open('/kaggle/working/data.yaml').read())

In [ ]:
# Cell 5 — quick run to produce best.pt (~15 min)
!yolo detect train \
  data=/kaggle/working/data.yaml \
  model=yolo11n.pt \
  epochs=10 \
  imgsz=480 \
  batch=32 \
  cache=ram \
  workers=4 \
  name=food_yolo11 \
  project=/kaggle/working/runs \
  exist_ok=True

In [ ]:
# Cell 6 — copy best.pt to working root (makes it available in kernel output)
import shutil, glob, os
matches = glob.glob('/kaggle/working/runs/**/best.pt', recursive=True)
if matches:
    shutil.copy(matches[0], '/kaggle/working/best.pt')
    size = os.path.getsize('/kaggle/working/best.pt') / 1e6
    print(f'best.pt saved ({size:.1f} MB)')
else:
    print('ERROR: best.pt not found')

In [ ]:
# Cell 7 — final metrics
import pandas as pd, glob
results = glob.glob('/kaggle/working/runs/**/results.csv', recursive=True)
if results:
    df = pd.read_csv(results[0])
    df.columns = df.columns.str.strip()
    cols = [c for c in ['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)'] if c in df.columns]
    print(df[cols].tail(10).to_string(index=False))